# Modul 06: Datenaufteilung, Verluste und Metriken

    **Notebooktyp:** Übungs- und Bewertungsnotebook  
    **Vorlesungen dieses Moduls:** Daten aufteilen, Verluste und Metriken  
    **Erwarteter Schwierigkeitsgrad:** Leicht fortgeschritten  
    **Orientierungszeit:** etwa 100 bis 135 Minuten

    ## Überblick

    Sie erstellen reproduzierbare zufällige, stratifizierte, gruppenbasierte und zeitliche Aufteilungen. Anschließend berechnen Sie Regressions- und Klassifikationsmetriken einschließlich Baselines, Schwellenwerten und Konfusionsmatrix.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_06A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_06B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Reproduzierbare Train-, Validierungs- und Testaufteilungen mit Indizes erstellen.
- Klassen, Gruppen und Zeitordnung bei Splits berücksichtigen.
- Typische Formen von Zielwert- und Vorverarbeitungsleckage erkennen.
- Regressionsverluste aus tatsächlichen Werten und Vorhersagen berechnen.
- Einfache Mittelwert-, Median- und Mehrheitsklassen-Baselines erstellen.
- Klassifikationsmetriken, Schwellenwerte und probabilistische Kennzahlen berechnen.

    ## Bewertete Fähigkeiten

    - Indexsplits, Stratifikation, GroupShuffleSplit und Zeitaufteilung
- disjunkte Mengen und Splitprotokolle prüfen
- MAE, MSE, RMSE und Baselines manuell berechnen
- Konfusionsmatrix, Precision, Recall, F1 und Log Loss verstehen
- Skalierung ausschließlich auf Trainingsdaten fitten

## Arbeitsanweisungen

Bearbeiten Sie die Aufgaben in der angegebenen Reihenfolge. Schreiben Sie Ihren Code ausschließlich in die klar markierten Arbeitszellen. Ergänzen Sie nach jeder Aufgabe eine kurze fachliche Reflexion. Verwenden Sie das Testset nicht für Modellwahl oder Hyperparameterentscheidungen, sofern die Aufgabe dies nicht ausdrücklich als abschließenden Schritt verlangt.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

X_classification, y_classification = make_classification(
    n_samples=240,
    n_features=5,
    n_informative=4,
    n_redundant=0,
    weights=[0.78, 0.22],
    class_sep=1.1,
    random_state=RANDOM_SEED,
)
classification_groups = np.repeat(np.arange(60), 4)
classification_data = pd.DataFrame(
    X_classification,
    columns=[f"feature_{i}" for i in range(X_classification.shape[1])],
)
classification_data["target"] = y_classification
classification_data["group_id"] = classification_groups

# Zeitlich geordnete Daten für eine kleine Prognoseaufgabe.
time_index = pd.date_range("2026-01-01", periods=180, freq="D")
time_feature = np.arange(180, dtype=float)
time_target = 20 + 0.08 * time_feature + 2.0 * np.sin(time_feature / 12) + rng.normal(0, 0.8, 180)
temporal_data = pd.DataFrame(
    {"date": time_index, "time_feature": time_feature, "target": time_target}
)

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Train-, Validierungs- und Testindizes manuell erzeugen

    Erzeugen Sie für 100 Beobachtungen reproduzierbare, zufällig gemischte Indizes mit 60 Prozent Training, 20 Prozent Validierung und 20 Prozent Test.

1. Verwenden Sie `np.random.default_rng(RANDOM_SEED)`.
2. Prüfen Sie die Größen der drei Mengen.
3. Prüfen Sie, dass keine Überschneidungen existieren.
4. Prüfen Sie, dass zusammen alle Indizes von 0 bis 99 genau einmal vorkommen.

> **Hinweis:** Prüfen Sie nicht nur die Längen, sondern auch Überschneidung und vollständige Abdeckung.

In [ ]:
n_samples = 100

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 1

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Prüfen Sie nicht nur die Längen, sondern auch Überschneidung und vollständige Abdeckung.

## Aufgabe 2: Stratifizierte, gruppenbasierte und zeitliche Splits vergleichen

    Erstellen Sie drei passende Aufteilungen.

1. Stratifizierter Train/Test-Split für `classification_data`, sodass die Klassenanteile ähnlich bleiben.
2. Gruppenbasierter Split, sodass keine `group_id` in Train und Test gleichzeitig vorkommt.
3. Zeitlicher Split von `temporal_data`, bei dem die ersten 75 Prozent Training und die letzten 25 Prozent Test bilden.

Erstellen Sie eine kleine Kontrolltabelle mit Größen, Klassenanteilen, Gruppenüberschneidung und Datumsbereichen.

> **Hinweis:** Wählen Sie den Split nach der Entstehungsstruktur der Beobachtungen, nicht nur nach Bequemlichkeit.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 2

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Wählen Sie den Split nach der Entstehungsstruktur der Beobachtungen, nicht nur nach Bequemlichkeit.

## Aufgabe 3: Datenleckage erkennen und Skalierung korrekt fitten

    Verwenden Sie `classification_data`.

1. Erzeugen Sie absichtlich ein Leckage-Merkmal `target_plus_noise = target + kleine Zufallsabweichung`.
2. Vergleichen Sie die Testgenauigkeit einer logistischen Regression mit und ohne dieses Merkmal auf demselben stratifizierten Split.
3. Skalieren Sie die legitimen Merkmale korrekt: `fit` nur auf Training, `transform` auf Training und Test.
4. Geben Sie Mittelwert und Standardabweichung der skalierten Trainings- und Testdaten aus.
5. Entfernen Sie das Leckage-Merkmal dauerhaft aus der Modellpipeline.

> **Hinweis:** Fragen Sie für jedes Merkmal: Wäre dieser Wert genau im Moment der Vorhersage bereits bekannt?

In [ ]:
leakage_data = classification_data.copy()

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 3

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Fragen Sie für jedes Merkmal: Wäre dieser Wert genau im Moment der Vorhersage bereits bekannt?

## Aufgabe 4: Regressionsverluste und Baselines manuell berechnen

    Verwenden Sie die unten angegebenen Trainings- und Testzielwerte.

1. Erzeugen Sie eine Mittelwert- und eine Median-Baseline aus `y_train_reg`.
2. Berechnen Sie für beide Baselines sowie für `model_predictions` MAE, MSE und RMSE ausschließlich mit NumPy.
3. Erstellen Sie eine sortierte Vergleichstabelle.
4. Zeigen Sie die Residuen des besten Verfahrens in einem Punktdiagramm.

> **Hinweis:** Berechnen Sie Baselinekonstanten nie aus dem Testsatz.

In [ ]:
y_train_reg = np.array([12, 14, 15, 15, 16, 18, 20, 22, 40], dtype=float)
y_test_reg = np.array([13, 16, 19, 24, 35], dtype=float)
model_predictions = np.array([14, 15, 20, 23, 30], dtype=float)

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 4

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Berechnen Sie Baselinekonstanten nie aus dem Testsatz.

## Aufgabe 5: Integrationsaufgabe: Schwellenwerte und Klassifikationsmetriken

    Gegeben sind wahre Labels und vorhergesagte Wahrscheinlichkeiten.

1. Erzeugen Sie Labels für Schwellenwerte 0.50 und 0.35.
2. Berechnen Sie TP, TN, FP und FN manuell.
3. Berechnen Sie Accuracy, Precision, Recall und F1 mit sicherer Behandlung möglicher Division durch null.
4. Berechnen Sie den probabilistischen Log Loss mit `sklearn.metrics.log_loss`.
5. Vergleichen Sie beide Schwellenwerte und empfehlen Sie einen für ein Szenario, in dem übersehene positive Fälle besonders teuer sind.

> **Hinweis:** Ein Schwellenwert ist eine Einsatzentscheidung und nicht zwingend ein fester Bestandteil des trainierten Modells.

In [ ]:
y_true = np.array([0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0])
positive_scores = np.array([0.08, 0.30, 0.42, 0.18, 0.77, 0.61, 0.48, 0.52, 0.12, 0.88, 0.36, 0.25])

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 5

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Ein Schwellenwert ist eine Einsatzentscheidung und nicht zwingend ein fester Bestandteil des trainierten Modells.

## Abschluss und Selbstkontrolle

Prüfen Sie vor der Abgabe, ob alle Arbeitszellen ausgefüllt sind, das Notebook von oben nach unten ohne unerwartete Fehler läuft, alle Diagramme beschriftet sind und jede Reflexion Ihre Beobachtungen sowie mindestens eine mögliche Fehlerquelle enthält.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.